# Notebook 4: Resistance Mutation Analysis

**Author:** Chidera Ibe (coibe2)

**Project:** Evolutionary Conservation of the PI3Kα Cryptic Drug Binding Site

---

In this notebook, we analyze the relationship between evolutionary conservation and clinically observed resistance mutations in PI3Kα (PIK3CA). We ask: do resistance mutations preferentially occur at positions that are less evolutionarily conserved? If so, this would support the hypothesis that highly conserved residues are under strong functional constraint and cannot easily mutate to confer drug resistance without compromising protein function.

**Key questions:**
1. Are resistance mutation sites significantly less conserved than other residues?
2. Are resistance mutations enriched in the cryptic drug binding pocket?
3. What do conservation patterns tell us about vulnerability to resistance?

---

## Section 1: Setup & Load Data

In [ ]:
# ---- Path setup ----
import sys
import os

# Add the project root to sys.path so we can import from src/
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# ---- Standard imports ----
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import mannwhitneyu, fisher_exact
import py3Dmol

# Project imports
from src.fetchers import fetch_pdb

# Plotting defaults
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

print("All imports successful.")

In [ ]:
# ---- Load conservation scores ----
conservation_path = os.path.join(PROJECT_ROOT, "data", "results", "conservation_scores.csv")
conservation_df = pd.read_csv(conservation_path)
print(f"Conservation scores loaded: {len(conservation_df)} positions")
print(f"Columns: {list(conservation_df.columns)}")
conservation_df.head()

In [ ]:
# ---- Load pocket residues ----
pocket_path = os.path.join(PROJECT_ROOT, "data", "results", "pocket_residues.csv")
pocket_df = pd.read_csv(pocket_path)
print(f"Pocket residues loaded: {len(pocket_df)} residues")
pocket_df.head()

In [ ]:
# Create a lookup dict: position -> conservation score
score_lookup = dict(zip(conservation_df["position"], conservation_df["conservation_score"]))

# Get the set of pocket positions for quick lookup
pocket_positions = set(pocket_df["position"].values) if "position" in pocket_df.columns else set()
print(f"Number of pocket positions: {len(pocket_positions)}")
print(f"Score lookup covers positions: {min(score_lookup.keys())} - {max(score_lookup.keys())}")

---

## Section 2: Resistance Mutation Data

### Clinical Context

**Varkaris et al., *Cancer Discovery* (2024)** reported secondary PIK3CA mutations observed in patients treated with PI3K inhibitors (e.g., alpelisib/inavolisib). These resistance mutations emerge under selective drug pressure and confer reduced drug sensitivity.

The key insight for our analysis: if the cryptic drug binding pocket is formed by evolutionarily conserved residues, then those residues may be resistant to mutation because changes would disrupt the protein's essential catalytic function. In contrast, resistance mutations may preferentially occur at positions that are less conserved, where mutations can alter drug binding without catastrophically disrupting kinase activity.

We define two classes of mutations:
- **Primary oncogenic (driver) mutations**: These activate PI3Kα and drive cancer. They are the reason patients receive PI3K inhibitors.
- **Secondary resistance mutations**: These emerge during treatment and reduce drug efficacy.

In [ ]:
# ---- Define mutation data ----

# Primary oncogenic (driver) mutations
PRIMARY_MUTATIONS = {
    "H1047R": {"position": 1047, "wt": "H", "mut": "R", "domain": "kinase"},
    "E545K":  {"position": 545,  "wt": "E", "mut": "K", "domain": "helical"},
    "E542K":  {"position": 542,  "wt": "E", "mut": "K", "domain": "helical"},
}

# Secondary resistance mutations (emerge under treatment)
RESISTANCE_MUTATIONS = {
    "W780R":  {"position": 780,  "wt": "W", "mut": "R", "domain": "kinase"},
    "Q859K":  {"position": 859,  "wt": "Q", "mut": "K", "domain": "kinase"},
    "Q859H":  {"position": 859,  "wt": "Q", "mut": "H", "domain": "kinase"},
    "E726K":  {"position": 726,  "wt": "E", "mut": "K", "domain": "C2"},
    "I817F":  {"position": 817,  "wt": "I", "mut": "F", "domain": "kinase"},
}

print(f"Primary oncogenic mutations: {len(PRIMARY_MUTATIONS)}")
print(f"Resistance mutations: {len(RESISTANCE_MUTATIONS)}")

In [ ]:
# ---- Look up conservation scores for each mutation ----

rows = []

for name, info in PRIMARY_MUTATIONS.items():
    pos = info["position"]
    cons = score_lookup.get(pos, np.nan)
    rows.append({
        "mutation": name,
        "position": pos,
        "wt": info["wt"],
        "mut": info["mut"],
        "domain": info["domain"],
        "type": "primary",
        "conservation_score": cons,
    })

for name, info in RESISTANCE_MUTATIONS.items():
    pos = info["position"]
    cons = score_lookup.get(pos, np.nan)
    rows.append({
        "mutation": name,
        "position": pos,
        "wt": info["wt"],
        "mut": info["mut"],
        "domain": info["domain"],
        "type": "resistance",
        "conservation_score": cons,
    })

mutations_df = pd.DataFrame(rows)
print("Combined mutation table:")
mutations_df

---

## Section 3: Statistical Analysis

We now test whether resistance mutations occur at positions with different conservation levels compared to the rest of the protein. We employ three complementary statistical approaches:

1. **Mann-Whitney U test** -- non-parametric comparison of conservation scores
2. **Permutation test** -- empirical null distribution for the mean conservation at resistance sites
3. **Fisher's exact test** -- enrichment of resistance mutations in the cryptic pocket

In [ ]:
# ---- Define residue groups ----

# Unique positions for each group
resistance_positions = list(set(info["position"] for info in RESISTANCE_MUTATIONS.values()))
primary_positions = list(set(info["position"] for info in PRIMARY_MUTATIONS.values()))
mutation_positions = set(resistance_positions + primary_positions)

# Conservation scores for each group
resistance_scores = np.array([score_lookup[p] for p in resistance_positions if p in score_lookup])
primary_scores = np.array([score_lookup[p] for p in primary_positions if p in score_lookup])
other_scores = np.array([
    score_lookup[p] for p in score_lookup
    if p not in mutation_positions
])

print(f"Resistance sites ({len(resistance_positions)} unique positions): {sorted(resistance_positions)}")
print(f"  Mean conservation: {np.mean(resistance_scores):.4f}")
print(f"Primary oncogenic sites ({len(primary_positions)} positions): {sorted(primary_positions)}")
print(f"  Mean conservation: {np.mean(primary_scores):.4f}")
print(f"All other residues: {len(other_scores)} positions")
print(f"  Mean conservation: {np.mean(other_scores):.4f}")

In [ ]:
# ---- Mann-Whitney U test ----
# Compare conservation at resistance sites vs. all other residues

stat, pval_mw = mannwhitneyu(resistance_scores, other_scores, alternative='two-sided')

print("Mann-Whitney U Test")
print("="*50)
print(f"  H0: Conservation at resistance sites = conservation at other sites")
print(f"  U statistic: {stat:.1f}")
print(f"  p-value: {pval_mw:.6f}")
print(f"  Significant at alpha=0.05? {'Yes' if pval_mw < 0.05 else 'No'}")

In [ ]:
# ---- Permutation test ----
# Test whether the mean conservation at resistance sites is significantly
# different from what we'd expect by chance.

np.random.seed(42)

n_resistance = len(resistance_positions)
all_scores = conservation_df["conservation_score"].values
observed_mean = np.mean(resistance_scores)

n_permutations = 10000
null_means = []
for _ in range(n_permutations):
    sample = np.random.choice(all_scores, size=n_resistance, replace=False)
    null_means.append(np.mean(sample))

null_means = np.array(null_means)

# p-value: fraction of null samples with mean <= observed
# (testing if resistance sites are LESS conserved)
p_perm = np.mean(null_means <= observed_mean)

print("Permutation Test (10,000 iterations)")
print("="*50)
print(f"  Observed mean conservation at resistance sites: {observed_mean:.4f}")
print(f"  Null distribution mean: {np.mean(null_means):.4f}")
print(f"  Null distribution std: {np.std(null_means):.4f}")
print(f"  p-value (less conserved): {p_perm:.4f}")
print(f"  Significant at alpha=0.05? {'Yes' if p_perm < 0.05 else 'No'}")

In [ ]:
# ---- Plot the null distribution ----

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(null_means, bins=50, color="steelblue", alpha=0.7, edgecolor="white", label="Null distribution")
ax.axvline(observed_mean, color="red", linewidth=2.5, linestyle="--",
           label=f"Observed mean = {observed_mean:.4f}")
ax.set_xlabel("Mean Conservation Score", fontsize=12)
ax.set_ylabel("Count", fontsize=12)
ax.set_title("Permutation Test: Mean Conservation at Resistance Sites\nvs. Random Sets of Residues",
             fontsize=13)
ax.legend(fontsize=11)
ax.text(0.02, 0.95, f"p = {p_perm:.4f}\nn = {n_permutations:,} permutations",
        transform=ax.transAxes, fontsize=10, verticalalignment="top",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow", alpha=0.8))
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "data", "results", "permutation_test.png"), dpi=150,
            bbox_inches="tight")
plt.show()
print("Saved: data/results/permutation_test.png")

In [ ]:
# ---- Fisher's exact test: resistance enrichment in cryptic pocket ----

# Total residues in the protein
total_residues = len(conservation_df)
n_pocket = len(pocket_positions)
n_non_pocket = total_residues - n_pocket

# Count resistance mutations in/out of pocket
resistance_in_pocket = sum(1 for p in resistance_positions if p in pocket_positions)
resistance_not_in_pocket = len(resistance_positions) - resistance_in_pocket

# Non-resistance residues in/out of pocket
non_resistance_in_pocket = n_pocket - resistance_in_pocket
non_resistance_not_in_pocket = n_non_pocket - resistance_not_in_pocket

# 2x2 contingency table
contingency = [
    [resistance_in_pocket, resistance_not_in_pocket],
    [non_resistance_in_pocket, non_resistance_not_in_pocket]
]

odds_ratio, pval_fisher = fisher_exact(contingency)

print("Fisher's Exact Test: Resistance Mutations in Cryptic Pocket")
print("="*60)
print(f"  Contingency table:")
print(f"                      In pocket    Not in pocket")
print(f"    Resistance:       {resistance_in_pocket:>8}     {resistance_not_in_pocket:>8}")
print(f"    Non-resistance:   {non_resistance_in_pocket:>8}     {non_resistance_not_in_pocket:>8}")
print(f"  Odds ratio: {odds_ratio:.4f}")
print(f"  p-value: {pval_fisher:.6f}")
print(f"  Significant at alpha=0.05? {'Yes' if pval_fisher < 0.05 else 'No'}")

In [ ]:
# ---- Effect size: rank-biserial correlation ----
# This measures how much the two distributions (resistance vs. other) differ
# r = 1 - (2U)/(n1*n2) where U is the Mann-Whitney U statistic

n1 = len(resistance_scores)
n2 = len(other_scores)
r_biserial = 1 - (2 * stat) / (n1 * n2)

print("Effect Size: Rank-Biserial Correlation")
print("="*50)
print(f"  r = {r_biserial:.4f}")
print(f"  Interpretation: {'Large' if abs(r_biserial) > 0.5 else 'Medium' if abs(r_biserial) > 0.3 else 'Small'} effect")
print(f"  Direction: resistance sites are {'less' if r_biserial < 0 else 'more'} conserved")

In [ ]:
# ---- Summary of all statistical tests ----

print("\n" + "="*65)
print("       STATISTICAL ANALYSIS SUMMARY")
print("="*65)
print(f"\n  Resistance sites (n={n1}):  mean conservation = {np.mean(resistance_scores):.4f}")
print(f"  Primary sites (n={len(primary_scores)}):     mean conservation = {np.mean(primary_scores):.4f}")
print(f"  Other residues (n={n2}): mean conservation = {np.mean(other_scores):.4f}")
print(f"\n  Mann-Whitney U test:       p = {pval_mw:.6f}  (U = {stat:.1f})")
print(f"  Permutation test:          p = {p_perm:.4f}")
print(f"  Fisher's exact test:       p = {pval_fisher:.6f}  (OR = {odds_ratio:.4f})")
print(f"  Rank-biserial correlation: r = {r_biserial:.4f}")
print("="*65)

---

## Section 4: Visualizations

We now create four visualizations to explore the relationship between conservation and resistance.

### 4a: Violin/Box Plot -- Conservation by Residue Group

Comparing conservation score distributions across four categories of residues.

In [ ]:
# ---- 4a: Violin + strip plot ----

# Assign each residue to a group
group_labels = []
group_scores = []

for _, row in conservation_df.iterrows():
    pos = row["position"]
    score = row["conservation_score"]
    if pos in resistance_positions:
        group_labels.append("Resistance\nsites")
    elif pos in primary_positions:
        group_labels.append("Primary oncogenic\nsites")
    elif pos in pocket_positions:
        group_labels.append("Pocket-lining\n(non-mutation)")
    else:
        group_labels.append("All other\nresidues")
    group_scores.append(score)

plot_df = pd.DataFrame({"Group": group_labels, "Conservation Score": group_scores})

# Define order and colors
order = ["Resistance\nsites", "Primary oncogenic\nsites", "Pocket-lining\n(non-mutation)", "All other\nresidues"]
palette = ["#e74c3c", "#3498db", "#2ecc71", "#95a5a6"]

fig, ax = plt.subplots(figsize=(10, 6))
sns.violinplot(data=plot_df, x="Group", y="Conservation Score", order=order,
               palette=palette, inner=None, alpha=0.4, ax=ax)
sns.stripplot(data=plot_df, x="Group", y="Conservation Score", order=order,
              palette=palette, size=3, alpha=0.6, jitter=True, ax=ax)
sns.boxplot(data=plot_df, x="Group", y="Conservation Score", order=order,
            color="white", width=0.15, showcaps=True, boxprops=dict(facecolor="none"),
            whiskerprops=dict(color="black"), medianprops=dict(color="black", linewidth=2),
            ax=ax)

# Add significance bracket if Mann-Whitney test is significant
if pval_mw < 0.05:
    y_max = plot_df["Conservation Score"].max()
    bracket_y = y_max + 0.05
    ax.plot([0, 0, 3, 3], [bracket_y, bracket_y + 0.02, bracket_y + 0.02, bracket_y],
            color="black", linewidth=1.2)
    sig_text = f"p = {pval_mw:.4f} *" if pval_mw < 0.05 else f"p = {pval_mw:.4f}"
    ax.text(1.5, bracket_y + 0.03, sig_text, ha="center", fontsize=10, fontweight="bold")

ax.set_title("Conservation Score Distribution by Residue Category", fontsize=14)
ax.set_ylabel("Conservation Score", fontsize=12)
ax.set_xlabel("")
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "data", "results", "conservation_by_group.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print("Saved: data/results/conservation_by_group.png")

### 4b: Conservation Landscape with Mutations

A position-by-position view of conservation scores across the entire PI3Kα sequence, with domain annotations and mutation positions highlighted.

In [ ]:
# ---- 4b: Conservation landscape ----

# Domain boundaries for PI3K-alpha (approximate, based on UniProt P42336)
DOMAINS = {
    "ABD":     (1, 108,    "#f0e68c"),   # Adaptor-binding domain
    "RBD":     (190, 291,  "#dda0dd"),   # Ras-binding domain
    "C2":      (330, 480,  "#87ceeb"),   # C2 domain
    "Helical": (525, 696,  "#98fb98"),   # Helical domain
    "Kinase":  (697, 1068, "#ffb6c1"),   # Kinase domain
}

fig, ax = plt.subplots(figsize=(14, 5))

# Background shading for domains
for domain_name, (start, end, color) in DOMAINS.items():
    ax.axvspan(start, end, alpha=0.15, color=color, label=domain_name)
    ax.text((start + end) / 2, ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 1.05,
            domain_name, ha="center", fontsize=8, fontstyle="italic", color="gray")

# Plot conservation scores as a line
positions = conservation_df["position"].values
scores = conservation_df["conservation_score"].values
ax.plot(positions, scores, color="gray", linewidth=0.5, alpha=0.6, zorder=1)
ax.scatter(positions, scores, c="lightgray", s=1, alpha=0.3, zorder=1)

# Overlay resistance mutations as red diamonds
for name, info in RESISTANCE_MUTATIONS.items():
    pos = info["position"]
    if pos in score_lookup:
        cons = score_lookup[pos]
        ax.scatter(pos, cons, marker="D", c="red", s=100, edgecolors="darkred",
                   linewidths=1.2, zorder=5)
        ax.annotate(name, (pos, cons), textcoords="offset points", xytext=(5, 8),
                    fontsize=8, fontweight="bold", color="red")

# Overlay primary mutations as blue squares
for name, info in PRIMARY_MUTATIONS.items():
    pos = info["position"]
    if pos in score_lookup:
        cons = score_lookup[pos]
        ax.scatter(pos, cons, marker="s", c="blue", s=100, edgecolors="darkblue",
                   linewidths=1.2, zorder=5)
        ax.annotate(name, (pos, cons), textcoords="offset points", xytext=(5, -12),
                    fontsize=8, fontweight="bold", color="blue")

# Domain labels at the top
y_top = max(scores) + 0.08
for domain_name, (start, end, color) in DOMAINS.items():
    ax.text((start + end) / 2, y_top, domain_name, ha="center", fontsize=9,
            fontstyle="italic", color="dimgray", fontweight="bold")

# Legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker="D", color="w", markerfacecolor="red", markersize=10,
           label="Resistance mutations"),
    Line2D([0], [0], marker="s", color="w", markerfacecolor="blue", markersize=10,
           label="Primary oncogenic mutations"),
]
ax.legend(handles=legend_elements, loc="lower right", fontsize=10)

ax.set_xlim(0, max(positions) + 10)
ax.set_xlabel("Residue Position", fontsize=12)
ax.set_ylabel("Conservation Score", fontsize=12)
ax.set_title("PI3K\u03b1 Conservation Landscape with Clinically Observed Mutations", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "data", "results", "conservation_landscape_mutations.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print("Saved: data/results/conservation_landscape_mutations.png")

### 4c: Structural View with Resistance Mutations

3D visualization of PI3Kα (PDB: 8TSD) showing the spatial locations of resistance and primary oncogenic mutations on the protein structure.

In [ ]:
# ---- 4c: py3Dmol structural visualization ----

# Fetch the PDB structure
pdb_text = fetch_pdb("8TSD")
print(f"PDB loaded: {len(pdb_text)} characters")

# Create the 3D view
view = py3Dmol.view(width=800, height=600)
view.addModel(pdb_text, "pdb")

# Base style: cartoon in light gray
view.setStyle({"chain": "A"}, {"cartoon": {"color": "lightgray", "opacity": 0.7}})

# Resistance mutations: large red spheres
for name, info in RESISTANCE_MUTATIONS.items():
    view.addStyle(
        {"resi": info["position"], "chain": "A"},
        {"sphere": {"color": "red", "radius": 1.5}}
    )

# Primary mutations: blue spheres
for name, info in PRIMARY_MUTATIONS.items():
    view.addStyle(
        {"resi": info["position"], "chain": "A"},
        {"sphere": {"color": "blue", "radius": 1.5}}
    )

# Ligand in yellow sticks
view.addStyle(
    {"hetflag": True, "not": {"resn": "HOH"}},
    {"stick": {"color": "yellow"}}
)

# Add labels for resistance mutations
for name, info in RESISTANCE_MUTATIONS.items():
    view.addLabel(name, {
        "fontSize": 12, "fontColor": "white", "backgroundColor": "red",
        "backgroundOpacity": 0.7, "position": {"resi": info["position"], "chain": "A"}
    })

# Add labels for primary mutations
for name, info in PRIMARY_MUTATIONS.items():
    view.addLabel(name, {
        "fontSize": 12, "fontColor": "white", "backgroundColor": "blue",
        "backgroundOpacity": 0.7, "position": {"resi": info["position"], "chain": "A"}
    })

view.zoomTo({"chain": "A"})
view.show()

**Figure legend:** PI3Kα structure (PDB: 8TSD) with resistance mutation sites shown as **red spheres** and primary oncogenic mutation sites as **blue spheres**. The bound ligand is shown as yellow sticks. The protein backbone is displayed as a semi-transparent gray cartoon.

### 4d: Conservation Comparison Bar Chart

Direct comparison of conservation scores at key resistance sites versus reference levels.

In [ ]:
# ---- 4d: Grouped bar chart ----

# Compute values for the bar chart
pocket_scores = np.array([score_lookup[p] for p in pocket_positions if p in score_lookup])
mean_pocket_cons = np.mean(pocket_scores)
mean_overall_cons = np.mean(all_scores)

bar_labels = []
bar_values = []
bar_colors = []

# Individual resistance sites
for pos in [780, 859]:
    label = f"W{pos}" if pos == 780 else f"Q{pos}"
    bar_labels.append(f"{label}\n(strong resistance)")
    bar_values.append(score_lookup.get(pos, 0))
    bar_colors.append("#e74c3c")

for pos in [726, 817]:
    label = f"E{pos}" if pos == 726 else f"I{pos}"
    bar_labels.append(f"{label}\n(resistance)")
    bar_values.append(score_lookup.get(pos, 0))
    bar_colors.append("#e67e22")

# Reference levels
bar_labels.append("Mean pocket\nconservation")
bar_values.append(mean_pocket_cons)
bar_colors.append("#2ecc71")

bar_labels.append("Mean overall\nconservation")
bar_values.append(mean_overall_cons)
bar_colors.append("#95a5a6")

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(range(len(bar_labels)), bar_values, color=bar_colors, edgecolor="white",
              linewidth=1.5, alpha=0.85)

# Add value labels on bars
for bar, val in zip(bars, bar_values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{val:.3f}", ha="center", fontsize=10, fontweight="bold")

ax.set_xticks(range(len(bar_labels)))
ax.set_xticklabels(bar_labels, fontsize=10)
ax.set_ylabel("Conservation Score", fontsize=12)
ax.set_title("Conservation at Resistance Sites vs. Reference Levels", fontsize=14)

# Add a horizontal line for overall mean
ax.axhline(mean_overall_cons, color="gray", linestyle="--", alpha=0.5, linewidth=1)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "data", "results", "resistance_conservation_bars.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print("Saved: data/results/resistance_conservation_bars.png")

---

## Section 5: Interpretation

### 5.1 Are resistance mutations at less-conserved positions?

Our statistical analyses above test whether clinically observed resistance mutations in PIK3CA tend to occur at positions with lower evolutionary conservation scores. If the Mann-Whitney U test and permutation test yield significant p-values, this supports the hypothesis that resistance preferentially arises at positions with more evolutionary flexibility.

This makes biological sense: a residue that is highly conserved across orthologous species is likely essential for the core catalytic function of PI3Kα. Mutating such a residue -- even if it would disrupt drug binding -- would also cripple the kinase activity that cancer cells depend on. Resistance mutations must walk a tightrope: disrupting drug binding while preserving enzymatic function.

### 5.2 The Evolutionary Constraint Hypothesis

The **evolutionary constraint hypothesis** predicts that:
- Residues under strong purifying selection (high conservation) cannot easily mutate without loss of function
- Drug binding sites that overlap with functionally constrained regions are therefore more resistant to the evolution of drug resistance
- If the cryptic pocket in PI3Kα is formed by highly conserved residues, drugs targeting this site may face fewer resistance escape routes

This is distinct from the standard pharmacological concept of drug resistance -- here, we are using evolutionary conservation as a *predictor* of resistance vulnerability.

### 5.3 The Selectivity Angle

An additional nuance emerges from considering paralog conservation versus ortholog conservation:
- **Ortholog conservation** (across species for the same gene): high conservation suggests functional constraint
- **Paralog divergence** (across PI3K isoforms within the same species): divergent positions can be exploited for isoform-selective drug design

The ideal drug target combines: (1) high ortholog conservation (constraining resistance) with (2) paralog divergence (enabling selectivity). The cryptic pocket in PI3Kα may exhibit exactly this pattern, offering a therapeutic advantage.

### 5.4 Clinical Implications

Based on conservation analysis, we can stratify residues by resistance vulnerability:
- **Low risk** (high conservation): Mutations here would likely disrupt kinase function. Drug contacts at these positions are more durable.
- **Moderate risk**: Some evolutionary flexibility exists, but constraints are still present.
- **High risk** (low conservation): These positions can mutate with minimal functional penalty, posing the greatest resistance threat.

This framework can help prioritize drug design efforts and anticipate resistance patterns.

### 5.5 Limitations

Several important caveats apply:
1. **Small sample size**: Only a handful of resistance mutations have been clinically characterized for PIK3CA. Our statistical power is limited.
2. **Conservation is necessary but not sufficient**: A conserved position resists mutation, but other factors (drug-residue contact geometry, allosteric effects, epistasis) also determine resistance.
3. **Ascertainment bias**: Clinically observed mutations may not represent all possible resistance mechanisms. Some mutations may be undiscovered.
4. **Structural context**: Conservation scores alone do not capture the 3D structural consequences of mutations.
5. **Single protein analysis**: Our conservation analysis is for PI3Kα only; resistance may also arise through other pathway components.

### 5.6 Future Directions

- **Molecular dynamics simulations**: Model the structural effects of resistance mutations on drug binding
- **Expanded resistance data**: As more patients are treated with PI3K inhibitors, additional resistance mutations will be characterized
- **Combination therapy strategies**: Use conservation analysis to identify complementary drug targets that constrain resistance escape routes
- **Machine learning**: Integrate conservation scores with other features (solvent accessibility, B-factors, contact maps) to build predictive models of resistance
- **Pan-cancer analysis**: Extend the framework to other kinases and drug targets

---

## Section 6: Summary Table

We create a comprehensive summary table combining all analyses, and export it for downstream use.

In [ ]:
# ---- Build the final summary DataFrame ----

# Determine domain for each position
def get_domain(pos):
    for domain_name, (start, end, _) in DOMAINS.items():
        if start <= pos <= end:
            return domain_name
    return "linker"

# Build a comprehensive table
summary_rows = []
resistance_pos_set = set(resistance_positions)
primary_pos_set = set(primary_positions)

for _, row in conservation_df.iterrows():
    pos = row["position"]
    summary_rows.append({
        "position": pos,
        "conservation_score": row["conservation_score"],
        "is_pocket": pos in pocket_positions,
        "is_resistance": pos in resistance_pos_set,
        "is_primary": pos in primary_pos_set,
        "domain": get_domain(pos),
    })

summary_df = pd.DataFrame(summary_rows)
print(f"Summary table: {len(summary_df)} rows x {len(summary_df.columns)} columns")
summary_df.head(10)

In [ ]:
# ---- Save the final analysis table ----

output_path = os.path.join(PROJECT_ROOT, "data", "results", "final_analysis.csv")
os.makedirs(os.path.dirname(output_path), exist_ok=True)
summary_df.to_csv(output_path, index=False)
print(f"Saved: {output_path}")

In [ ]:
# ---- Print key findings ----

print("\n" + "="*65)
print("              KEY FINDINGS")
print("="*65)
print()
print(f"  * Total residues analyzed: {len(summary_df)}")
print(f"  * Pocket residues: {summary_df['is_pocket'].sum()}")
print(f"  * Resistance mutation sites: {summary_df['is_resistance'].sum()}")
print(f"  * Primary oncogenic sites: {summary_df['is_primary'].sum()}")
print()
print(f"  * Mean conservation (all residues): {summary_df['conservation_score'].mean():.4f}")
print(f"  * Mean conservation (pocket): "
      f"{summary_df.loc[summary_df['is_pocket'], 'conservation_score'].mean():.4f}")
print(f"  * Mean conservation (resistance sites): "
      f"{summary_df.loc[summary_df['is_resistance'], 'conservation_score'].mean():.4f}")
print(f"  * Mean conservation (primary sites): "
      f"{summary_df.loc[summary_df['is_primary'], 'conservation_score'].mean():.4f}")
print()
print(f"  * Mann-Whitney p-value (resistance vs. other): {pval_mw:.6f}")
print(f"  * Permutation test p-value: {p_perm:.4f}")
print(f"  * Fisher's exact test p-value (pocket enrichment): {pval_fisher:.6f}")
print(f"  * Effect size (rank-biserial r): {r_biserial:.4f}")
print()
print(f"  * Resistance sites in pocket: {resistance_in_pocket}/{len(resistance_positions)}")
print(f"  * Output saved to: data/results/final_analysis.csv")
print()
print("="*65)
print("  Analysis complete.")
print("="*65)